In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,   # ADD THIS
    confusion_matrix
)

from xgboost import XGBClassifier

import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# ==========================================================
# LOAD DATA
# ==========================================================

df = pd.read_csv("mimic_multidisease_ehr.csv")

# ----------------------------------------------------------
# FEATURES
# ----------------------------------------------------------

feature_cols = [
    'Age',
    'Gender',

    'heart_rate_min',
    'heart_rate_max',
    'heart_rate_avg',

    'sys_bp_min',
    'sys_bp_max',
    'sys_bp_avg',

    'resp_rate_min',
    'resp_rate_max',
    'resp_rate_avg',

    'spo2_min',

    'wbc_max',
    'creatinine_max'
]

target_cols = [
    'Sepsis_Label',
    'AKI_Label',
    'CHF_Label',
    'Diabetes_Label'
]

X = df[feature_cols]
Y = df[target_cols]

# ==========================================================
# TRAIN TEST SPLIT
# ==========================================================

X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.20,
    random_state=42
)

print(f"Training Patients : {len(X_train):,}")
print(f"Testing Patients  : {len(X_test):,}")

# ==========================================================
# MODEL
# ==========================================================

base_model = XGBClassifier(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=5,
    tree_method='hist',
    eval_metric='logloss',
    random_state=42
)

model = MultiOutputClassifier(
    base_model,
    n_jobs=-1
)

print("\nTraining model...")
model.fit(X_train, Y_train)

# ==========================================================
# PROBABILITIES
# ==========================================================

probabilities = model.predict_proba(X_test)

prob_matrix = np.column_stack([
    probabilities[i][:,1]
    for i in range(len(target_cols))
])

# ==========================================================
# THRESHOLD OPTIMIZATION
# ==========================================================

thresholds = np.arange(0.05, 1.00, 0.05)

results = []

print("\n========== THRESHOLD CALIBRATION ==========\n")

for i, disease in enumerate(target_cols):

    y_true = Y_test.iloc[:, i]
    y_prob = prob_matrix[:, i]

    roc_auc = roc_auc_score(y_true, y_prob)
    pr_auc = average_precision_score(y_true, y_prob)

    best_f1 = 0
    best_threshold = 0.5

    best_precision = 0
    best_recall = 0

    f1_curve = []

    for threshold in thresholds:

        y_pred = (y_prob >= threshold).astype(int)

        precision = precision_score(
            y_true,
            y_pred,
            zero_division=0
        )

        recall = recall_score(
            y_true,
            y_pred,
            zero_division=0
        )

        f1 = f1_score(
            y_true,
            y_pred,
            zero_division=0
        )

        f1_curve.append(f1)

        if f1 > best_f1:

            best_f1 = f1
            best_threshold = threshold
            best_precision = precision
            best_recall = recall

    results.append([
        disease,
        roc_auc,
        pr_auc,
        best_threshold,
        best_precision,
        best_recall,
        best_f1
    ])

    # ------------------------------------------------------
    # Plot Threshold vs F1
    # ------------------------------------------------------

    plt.figure(figsize=(7,5))

    plt.plot(thresholds, f1_curve)

    plt.axvline(
        best_threshold,
        linestyle='--'
    )

    plt.title(f"{disease} Threshold Optimization")

    plt.xlabel("Threshold")

    plt.ylabel("F1 Score")

    plt.tight_layout()

    plt.savefig(
        f"{disease}_threshold_curve.png",
        dpi=300
    )

    plt.close()

# ==========================================================
# RESULTS TABLE
# ==========================================================

results_df = pd.DataFrame(
    results,
    columns=[
        "Disease",
        "ROC_AUC",
        "PR_AUC",
        "Optimal_Threshold",
        "Precision",
        "Recall",
        "F1"
    ]
)

print(results_df.round(3))

results_df.to_csv(
    "threshold_calibration_results.csv",
    index=False
)

print("\nSaved:")
print(" - threshold_calibration_results.csv")
print(" - *_threshold_curve.png")

Training Patients : 41,902
Testing Patients  : 10,476

Training model...

========== THRESHOLD CALIBRATION ==========

          Disease  ROC_AUC  PR_AUC  Optimal_Threshold  Precision  Recall  \
0    Sepsis_Label    0.820   0.484               0.25      0.440   0.583   
1       AKI_Label    0.885   0.770               0.40      0.718   0.738   
2       CHF_Label    0.758   0.489               0.25      0.405   0.679   
3  Diabetes_Label    0.683   0.434               0.25      0.362   0.734   

      F1  
0  0.501  
1  0.728  
2  0.507  
3  0.485  

Saved:
 - threshold_calibration_results.csv
 - *_threshold_curve.png


In [2]:
# ==========================================================
# DISEASE-SPECIFIC THRESHOLDS
# ==========================================================

thresholds = {
    'Sepsis_Label': 0.25,
    'AKI_Label': 0.40,
    'CHF_Label': 0.25,
    'Diabetes_Label': 0.25
}

print("\n========== FINAL CLINICAL SCREENING RESULTS ==========\n")

for i, disease in enumerate(target_cols):

    y_true = Y_test.iloc[:, i]
    y_prob = prob_matrix[:, i]

    threshold = thresholds[disease]

    y_pred = (y_prob >= threshold).astype(int)

    print(f"\n===== {disease} =====")
    print(f"Threshold Used: {threshold}")

    print(classification_report(
        y_true,
        y_pred,
        digits=3
    ))

    print("Confusion Matrix")
    print(confusion_matrix(y_true, y_pred))


========== FINAL CLINICAL SCREENING RESULTS ==========


===== Sepsis_Label =====
Threshold Used: 0.25
              precision    recall  f1-score   support

           0      0.916     0.860     0.887      8813
           1      0.440     0.583     0.501      1663

    accuracy                          0.816     10476
   macro avg      0.678     0.721     0.694     10476
weighted avg      0.840     0.816     0.826     10476

Confusion Matrix
[[7578 1235]
 [ 694  969]]

===== AKI_Label =====
Threshold Used: 0.4
              precision    recall  f1-score   support

           0      0.888     0.877     0.883      7359
           1      0.718     0.738     0.728      3117

    accuracy                          0.836     10476
   macro avg      0.803     0.808     0.805     10476
weighted avg      0.837     0.836     0.837     10476

Confusion Matrix
[[6456  903]
 [ 816 2301]]

===== CHF_Label =====
Threshold Used: 0.25
              precision    recall  f1-score   support

           0